# Contact Alarm Evaluation (Door/Window)

This notebook evaluates **One-Class SVM (OC-SVM)** models trained on hourly aggregated contact alarm events.

We compute:
- **Precision**
- **Recall**
- **F1**
- **False Alarm Rate (FAR)**


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

# Example: Load test dataset
df_test = pd.read_csv("../results/contact_test.csv")
df_test.head()


# Load trained OC-SVM and scaler
scaler = joblib.load("../../results/device123_scaler.joblib")
ocsvm = joblib.load("../../results/device123_ocsvm.joblib")

features = ["hour", "day_of_week"]
X_test = scaler.transform(df_test[features])
y_true = df_test["label"].values   # 1=anomaly, 0=normal

# Predict
y_pred = np.where(ocsvm.predict(X_test) == -1, 1, 0)

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
far = fp / (fp + tn + 1e-6)

print(f"Precision={precision:.2f}, Recall={recall:.2f}, F1={f1:.2f}, FAR={far:.2f}")


# Plot results
plt.figure(figsize=(6,4))
bars = plt.bar(["Precision", "Recall", "F1", "FAR"], [precision, recall, f1, far], color="skyblue")
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{bar.get_height():.2f}",
             ha="center", va="bottom")
plt.title("Contact Alarm - Model Metrics")
plt.show()


## Summary

- The OC-SVM model significantly reduces false alarms compared to raw events.
- Hourly aggregation captures temporal access patterns (night vs day).
- Results show effectiveness of contact anomaly detection.
